# L5 · Notebook 04 — TD(0) 是 RM 算法

**对应 PPT**：`L5-SA.tex` ★ TD(0) 是 RM 算法（line 563）；L7 也会重做

## 教学目标

把 TD(0) 套进 RM 框架，验证：

1. 取 $g(V) = V - T^\pi V$（贝尔曼期望算子的不动点方程）
2. 带噪样本 $\widetilde g = V(s) - [R + \gamma V(s')]$（单步采样）
3. 验证 RM 的 (C2)：$\E[\eta|s] = 0$（贝尔曼算子是采样的期望）
4. 验证 (C3)：$T^\pi$ 是 $\gamma$-压缩 ⇒ $g$ monotone
5. 5-state random walk 实测 TD(0) 收敛

## 结论

**TD(0) 收敛性 = RM 收敛定理在 $g=V-T^\pi V$ 上的应用**——这是 L5 与 L7 的核心桥梁。

本 notebook 中的算法实现 = L7 `td_vs_mc_demo.py` 的简化版（去掉 MC 对比）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sutton-Barto 例 6.2: 5-state random walk
# 状态 0..6（0/6 终止，5 个内部状态）；行为：等概率左/右；右终止 +1，左终止 0
# π = uniform random；γ = 1
# 真值 V^π(s) = s/6 for s in 1..5
V_true = np.array([0, 1/6, 2/6, 3/6, 4/6, 5/6, 0])
gamma = 1.0

def episode(seed):
    rng = np.random.default_rng(seed)
    s = 3
    traj = []
    while True:
        s_next = s + (1 if rng.random() < 0.5 else -1)
        if s_next == 6:
            traj.append((s, 1.0, s_next, True))
            return traj
        if s_next == 0:
            traj.append((s, 0.0, s_next, True))
            return traj
        traj.append((s, 0.0, s_next, False))
        s = s_next

## 1. RM 框架对应（教学诊断）

| RM 元素 | TD(0) 对应 |
|---|---|
| $g(V) = ?$ | $V - T^\pi V$（贝尔曼期望算子的不动点方程）|
| 单样本估计 $\widetilde g$ | $V(s) - [R + \gamma V(s')]$（一步采样 transition）|
| 噪声 $\eta = \widetilde g - g$ | $(T^\pi V)(s) - [R + \gamma V(s')]$ |
| (C2) 噪声零均值 | $\E[R + \gamma V(s')|s] = (T^\pi V)(s)$ ✓ |
| (C3) monotone | $T^\pi$ 是 $\gamma$-压缩 ($\gamma\le 1$) ⇒ $g$ Lipschitz、有唯一零点 $V^\pi$ |
| 步长 (C1) | $\alpha_k\to 0$ 满足 RM 步长 |
| RM 更新 | $V(s)\leftarrow V(s) - \alpha_k[V(s) - R - \gamma V(s')]$ |

**最后一行就是 TD(0) 经典写法**：$V(s)\leftarrow V(s) + \alpha[R + \gamma V(s') - V(s)]$（把 $g$ 的负号吸进去）。

## 2. TD(0) 收敛实测

In [ ]:
def td0(n_episodes, alpha_fn, seed=0):
    V = np.zeros(7)  # 含两端终止态
    rmse_history = []
    for ep in range(n_episodes):
        for s, r, s_next, done in episode(seed * 1000 + ep):
            V_next = 0 if done else V[s_next]
            alpha = alpha_fn(ep)
            V[s] = V[s] + alpha * (r + gamma * V_next - V[s])
        rmse = np.sqrt(np.mean((V[1:6] - V_true[1:6])**2))
        rmse_history.append(rmse)
    return V, np.array(rmse_history)

n_eps = 200
V1, err1 = td0(n_eps, lambda ep: 0.1)
V2, err2 = td0(n_eps, lambda ep: 1.0/(ep+1))
V3, err3 = td0(n_eps, lambda ep: 0.05/(ep+1)**2)  # 破坏 (C1.1)

print(f'真值        : {V_true[1:6].round(4)}')
print(f'α=0.1 常数  : {V1[1:6].round(4)}  RMSE={err1[-1]:.4f}')
print(f'α=1/(k+1)   : {V2[1:6].round(4)}  RMSE={err2[-1]:.4f}')
print(f'α=0.05/k²   : {V3[1:6].round(4)}  RMSE={err3[-1]:.4f}（破坏 RM (C1.1)，应卡住）')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(err1, label=r'$\alpha=0.1$ const', color='C1')
ax.semilogy(err2, label=r'$\alpha=1/(k+1)$ (RM 合规)', color='C2')
ax.semilogy(err3, label=r'$\alpha=0.05/(k+1)^2$ (破 RM C1.1)', color='C3')
ax.set_xlabel('episode'); ax.set_ylabel(r'$\sqrt{\mathrm{MSE}(\hat V, V^\pi)}$')
ax.set_title('TD(0) on 5-state random walk —— 步长选择对收敛的影响（验证 RM (C1)）')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.savefig('figures/td0_as_rm.png', dpi=120, bbox_inches='tight'); plt.show()

## 3. 课堂诊断小结

| 论断 | 数值证据 |
|---|---|
| TD(0) = RM with $g=V-T^\pi V$ | RM 推导式 = TD(0) 经典更新式 |
| RM (C2) 噪声零均值 | $\E[R+\gamma V(s')\|s]=(T^\pi V)(s)$ |
| RM (C3) monotone | $T^\pi$ $\gamma$-压缩 ⇒ $g$ 单调有唯一零点 |
| 步长破 (C1) ⇒ TD 失败 | $\alpha=0.05/k^2$ 卡住远离真值 |

## 工程意义

- **TD(0) 收敛性**完全继承自 RM——这就是 L5 在 RL 课程中存在的意义
- **L7 Q-learning**：把 $T^\pi$ 换成 $T^*$，RM 推导原样套用，加上 off-policy 仍收敛
- **L8 deadly triad**：函数逼近 + bootstrap + off-policy 破坏 RM (C3)，是 RM 失效的典型场景

## 思考题

1. 把 $\gamma$ 改为 1.0，TD(0) 的 (C3) 是否仍成立？（提示：$\gamma=1$ 时 $T^\pi$ 不再严格压缩，需 episodic 任务）
2. TD(0) 在 5-state random walk 上的渐近偏差是 0；改成 $\alpha=$ 常数时偏差呢？
3. 把 TD(0) 改为 TD(λ)（多步），RM 框架如何套？(C2) 的噪声方差会如何变化？